# Nested feature-selection revision

This notebook audits the continuation of feature-selection-2.2 without rewriting its frozen negative result. Inner feature ranking uses 2017–2020 only, while 2021–2022 is reserved for outer feature-count and stopping decisions.


## Locate nested artifacts

The canonical runner writes the nested revision under a separate artifact root. Until the full run completes, this notebook reports that state without falling back to the original test-consumed selection.

In [1]:
from pathlib import Path
import json

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "data" / "splits").is_dir():
        PROJECT_ROOT = candidate
        break
EXP_DIR = PROJECT_ROOT / "notebooks/experiment/derived_8.2-feature-selection-2.2"
ARTIFACT_ROOT = EXP_DIR / "artifacts/nested"
print("Nested artifacts available:", ARTIFACT_ROOT.exists())

Nested artifacts available: True


## Compare inner and outer decisions

The outer table is the key safeguard added after the first 2.2 run. Candidate lists are generated entirely inside the train period, then scored on disjoint future years and held-out station groups.

In [2]:
paths = sorted(ARTIFACT_ROOT.glob("*/global/selected_features.json"))
rows = []
for path in paths:
    payload = json.loads(path.read_text())
    for candidate in payload["outer_candidate_summaries"]:
        rows.append({
            "dataset": payload["dataset"],
            "selected_n": payload["n_features"],
            "candidate_n": candidate["n_features"],
            "outer_mean_nrmse": candidate["mean_nrmse"],
            "outer_ucb": candidate["upper_confidence_bound"],
        })
if rows:
    display(pd.DataFrame(rows).sort_values(["dataset", "outer_ucb"]))
else:
    print("Run run_nested_selection.py to produce the full nested audit.")

,dataset,selected_n,candidate_n,outer_mean_nrmse,outer_ucb
6,derived_8.0,40,40,0.485542,0.502409
5,derived_8.0,40,50,0.497424,0.514115
1,derived_8.0,40,125,0.522524,0.548394
4,derived_8.0,40,65,0.526772,0.548668
2,derived_8.0,40,100,0.528205,0.553266
3,derived_8.0,40,80,0.531074,0.555244
0,derived_8.0,40,150,0.556591,0.591315
11,derived_8.2,65,65,0.722580,0.754889
10,derived_8.2,65,80,0.725372,0.758267
12,derived_8.2,65,50,0.726924,0.758699


## Locked learner, crossed folds, and MoE ablation

The architectural diagnostics below are retrospective only. They compare the locked final learner, independently generated forward-time and station/time paths, progressive elimination, and the shared-only versus shared-plus-delta MoE. No feature list is seeded or bypassed.

In [3]:
diagnostic_roots = {
    "joint nested": ARTIFACT_ROOT / "candidate_diagnostics/global_candidates.csv",
    "crossed": EXP_DIR / "artifacts/crossed_candidates_locked_outer/candidate_diagnostics/global_candidates.csv",
    "progressive": EXP_DIR / "artifacts/progressive_crossed_locked_outer/candidate_diagnostics/global_candidates.csv",
}
candidate_frames = []
for path_name, path in diagnostic_roots.items():
    if path.exists():
        frame = pd.read_csv(path)
        frame.insert(0, "path", path_name)
        candidate_frames.append(frame)
candidates = pd.concat(candidate_frames, ignore_index=True)
ceiling = (
    candidates[candidates["beta"].eq(0.0)]
    .sort_values("retrospective_test_R2", ascending=False)
    .groupby(["dataset", "path"], as_index=False)
    .first()[["dataset", "path", "n_features", "outer_R2", "retrospective_test_R2"]]
)
display(ceiling.sort_values(["dataset", "retrospective_test_R2"], ascending=[True, False]))

,dataset,path,n_features,outer_R2,retrospective_test_R2
2,derived_8.0,progressive,80,0.840849,0.820916
0,derived_8.0,crossed,40,0.831181,0.800203
1,derived_8.0,joint nested,150,0.828883,0.776693
3,derived_8.2,crossed,80,0.758540,0.635066
4,derived_8.2,joint nested,80,0.758540,0.635066


In [4]:
moe_path = ARTIFACT_ROOT / "retrospective_test_eval/metrics_summary.csv"
moe = pd.read_csv(moe_path)
models = [
    "2.2_global",
    "2.2_clustering_frozen_k2_shared_only",
    "2.2_clustering_dynamic_k2_shared_plus_delta",
    "2.2_clustering_refit_k2_shared_plus_delta",
]
display(
    moe[moe["dataset"].eq("derived_8.2") & moe["model"].isin(models)]
    .loc[:, ["model", "beta", "R2", "RMSE", "Bias"]]
    .sort_values(["beta", "R2"], ascending=[True, False])
)

,model,beta,R2,RMSE,Bias
4,2.2_global,0.0,0.635160,0.063604,-0.013919
7,2.2_clustering_dynamic_k2_shared_plus_delta,0.0,0.627913,0.064233,-0.013003
8,2.2_clustering_frozen_k2_shared_only,0.0,0.627913,0.064233,-0.013003
9,2.2_clustering_refit_k2_shared_plus_delta,0.0,0.625158,0.064470,-0.013420
10,2.2_global,0.2,0.621239,0.064806,-0.014310
13,2.2_clustering_dynamic_k2_shared_plus_delta,0.2,0.618748,0.065019,-0.014581
14,2.2_clustering_frozen_k2_shared_only,0.2,0.618748,0.065019,-0.014581
15,2.2_clustering_refit_k2_shared_plus_delta,0.2,0.610836,0.065690,-0.014593
